In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Training Hierarchical LightGBM Pipeline Layers (`models/train_hierarchical_lightgbm_layers.ipynb`)

This notebook trains the **3-Layer Hierarchical LightGBM Triage Model Architecture** on 35 predictor features, generating model artifacts and performance reports for each layer:

### 3-Layer Hierarchical Pipeline Architecture
1. **Layer 1 LightGBM (ESI 1 vs Non-ESI 1 Detector)**:
   - **Training Data**: Trained on **complete data** (`train_df`, 546,862 samples).
   - **Target**: `1` if ESI 1, `0` if Non-ESI 1.
   - **Hyperparameters**: `objective = "binary"`, `is_unbalance = TRUE`, `learning_rate = 0.05`, `num_leaves = 31`, `nrounds = 100`.
   - **Artifact Export**: Saved to `deploy/lightgbm_layer1_esi1_model.rds`.
2. **Layer 2 LightGBM (ESI 2/3 vs ESI 4/5 Specialist)**:
   - **Training Data**: Trained **without ESI 1 rows** (`train_df %>% filter(target_col != "1")`).
   - **Target**: `1` if ESI 2 or 3 (`"2_3"`), `0` if ESI 4 or 5 (`"4_5"`).
   - **Hyperparameters**: `objective = "binary"`, `learning_rate = 0.05`, `num_leaves = 31`, `nrounds = 100`.
   - **Artifact Export**: Saved to `deploy/rf_esi23_esi45_extreme_model.rds`.
3. **Layer 3A & 3B LightGBM Specialists**:
   - **Layer 3A (ESI 2 vs ESI 3)**: Trained strictly on ESI 2 & 3 rows (`deploy/lightgbm_esi23_model.rds`).
   - **Layer 3B (ESI 4 vs ESI 5)**: Trained strictly on ESI 4 & 5 rows (`deploy/lightgbm_esi45_model.rds`).

### Evaluation Metrics
Reports **Recall (Sensitivity)**, **Specificity**, **Balanced Accuracy**, and **ROC-AUC** for Validation and Test splits in `reports/`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(caret)
  library(dplyr)
  library(ggplot2)
  library(tidyr)
  library(pROC)
  library(lightgbm)
})
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) config_path <- "config/triage_conf.json"
config <- fromJSON(config_path)
cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Val Size:        ", config$training$val_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Data & Construct 35 Predictor Features
# ---------------------------------------------------------
set.seed(config$training$random_state)
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) data_file <- paste0("../", data_file)
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
raw_df <- get(data_obj_name, envir = data_env)
target_col_name <- config$classes$target_col
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
p_last   <- get_vec("pulse_last"); p_max    <- get_vec("pulse_max"); p_min    <- get_vec("pulse_min")
s_last   <- get_vec("sbp_last");   s_max    <- get_vec("sbp_max");   s_min    <- get_vec("sbp_min")
o2_last  <- get_vec("spo2_last");  o2_max   <- get_vec("spo2_max");  o2_min   <- get_vec("spo2_min")
r_last   <- get_vec("resp_last");   r_max    <- get_vec("resp_max");   r_min    <- get_vec("resp_min")
t_hr     <- get_vec("triage_vital_hr"); t_sbp <- get_vec("triage_vital_sbp"); t_o2 <- get_vec("triage_vital_o2"); t_rr <- get_vec("triage_vital_rr")
df_full <- data.frame(
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  triage_vital_hr         = t_hr,
  triage_vital_sbp        = t_sbp,
  triage_vital_rr         = t_rr,
  triage_vital_o2         = t_o2,
  pulse_last              = p_last,
  resp_last               = r_last,
  spo2_last               = o2_last,
  sbp_last                = s_last,
  pulse_min               = p_min,
  resp_min                = r_min,
  spo2_min                = o2_min,
  sbp_min                 = s_min,
  pulse_max               = p_max,
  resp_max                = r_max,
  spo2_max                = o2_max,
  sbp_max                 = s_max,
  hr_mean_to_last         = t_hr - p_last,
  sbp_mean_to_last        = t_sbp - s_last,
  spo2_mean_to_last       = t_o2 - o2_last,
  rr_mean_to_last         = t_rr - r_last,
  hr_range                = p_max - p_min,
  rr_range                = r_max - r_min,
  spo2_range              = o2_max - o2_min,
  sbp_range               = s_max - s_min,
  hr_last_to_min          = p_last - p_min,
  rr_last_to_min          = r_last - r_min,
  spo2_last_to_min        = o2_last - o2_min,
  sbp_last_to_min         = s_last - s_min,
  hr_last_to_max          = p_last - p_max,
  rr_last_to_max          = r_last - r_max,
  spo2_last_to_max        = o2_last - o2_max,
  sbp_last_to_max         = s_last - s_max
)
raw_esi <- as.character(raw_df[[target_col_name]])
df_full$target_col <- factor(raw_esi, levels = c("1", "2", "3", "4", "5"))
df_full <- na.omit(df_full)
test_size <- config$training$test_size
val_size  <- config$training$val_size
in_train_val <- createDataPartition(df_full$target_col, p = 1 - test_size, list = FALSE)
train_val_df <- df_full[in_train_val, ]
test_df      <- df_full[-in_train_val, ]
rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df$target_col, p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]
binary_cols <- c("gender", "cc_breathingdifficulty")
cont_cols   <- setdiff(names(train_df), c(binary_cols, "target_col"))
preproc <- preProcess(train_df[, cont_cols, drop = FALSE], method = c("center", "scale"))
train_scaled <- predict(preproc, train_df)
val_scaled   <- predict(preproc, val_df)
test_scaled  <- predict(preproc, test_df)
feat_names <- setdiff(names(train_scaled), "target_col")
cat(sprintf("Partitions Prepared: Train=%d, Val=%d, Test=%d\n", nrow(train_scaled), nrow(val_scaled), nrow(test_scaled)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Train Layer 1 LightGBM (ESI 1 vs Non-ESI 1 Binary Classifier - Complete Data)
# ---------------------------------------------------------
set.seed(config$training$random_state)
# Target: 1 if ESI 1, 0 if Non-ESI 1
y_tr_l1 <- ifelse(train_scaled$target_col == "1", 1, 0)
y_vl_l1 <- ifelse(val_scaled$target_col == "1", 1, 0)
y_ts_l1 <- ifelse(test_scaled$target_col == "1", 1, 0)
dtrain_l1 <- lgb.Dataset(data = as.matrix(train_scaled[, feat_names]), label = y_tr_l1)
dval_l1   <- lgb.Dataset(data = as.matrix(val_scaled[, feat_names]),   label = y_vl_l1)
params_l1 <- list(
  objective        = "binary",
  metric           = "binary_logloss",
  is_unbalance     = TRUE,
  learning_rate    = 0.05,
  num_leaves       = 31,
  max_depth        = 6,
  feature_fraction = 0.8,
  bagging_fraction = 0.8,
  bagging_freq     = 1
)
lgb_l1_model <- lgb.train(
  params                = params_l1,
  data                  = dtrain_l1,
  nrounds               = 100,
  valids                = list(train = dtrain_l1, val = dval_l1),
  early_stopping_rounds = 10,
  verbose               = 0
)
# Evaluate Layer 1 performance
p_l1_val  <- predict(lgb_l1_model, as.matrix(val_scaled[, feat_names]))
p_l1_test <- predict(lgb_l1_model, as.matrix(test_scaled[, feat_names]))
auc_l1_val  <- as.numeric(pROC::roc(y_vl_l1, p_l1_val)$auc)
auc_l1_test <- as.numeric(pROC::roc(y_ts_l1, p_l1_test)$auc)
cm_l1 <- confusionMatrix(factor(ifelse(p_l1_test >= 0.5, 1, 0)), factor(y_ts_l1))
rec_l1  <- cm_l1$byClass["Sensitivity"]
spec_l1 <- cm_l1$byClass["Specificity"]
bal_l1  <- cm_l1$byClass["Balanced Accuracy"]
cat("============================================================\n")
cat("   LAYER 1 LIGHTGBM (ESI 1 vs NON-ESI 1) TEST REPORT\n")
cat("============================================================\n")
cat(sprintf("  ESI 1 Sensitivity (Recall) : %.4f\n", rec_l1))
cat(sprintf("  Non-ESI 1 Specificity      : %.4f\n", spec_l1))
cat(sprintf("  Balanced Accuracy          : %.4f\n", bal_l1))
cat(sprintf("  ROC-AUC (Test)             : %.4f\n", auc_l1_test))
cat("============================================================\n\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Train Layer 2 LightGBM (ESI 2/3 vs ESI 4/5 Specialist - Without ESI 1 Rows)
# ---------------------------------------------------------
set.seed(config$training$random_state)
# Filter training & validation data WITHOUT ESI 1 rows
train_l2_df <- train_scaled %>% filter(target_col != "1")
val_l2_df   <- val_scaled   %>% filter(target_col != "1")
test_l2_df  <- test_scaled  %>% filter(target_col != "1")
# Target: 1 if ESI 2 or 3 ('2_3'), 0 if ESI 4 or 5 ('4_5')
y_tr_l2 <- ifelse(train_l2_df$target_col %in% c("2", "3"), 1, 0)
y_vl_l2 <- ifelse(val_l2_df$target_col %in% c("2", "3"), 1, 0)
y_ts_l2 <- ifelse(test_l2_df$target_col %in% c("2", "3"), 1, 0)
dtrain_l2 <- lgb.Dataset(data = as.matrix(train_l2_df[, feat_names]), label = y_tr_l2)
dval_l2   <- lgb.Dataset(data = as.matrix(val_l2_df[, feat_names]),   label = y_vl_l2)
params_l2 <- list(
  objective        = "binary",
  metric           = "binary_logloss",
  learning_rate    = 0.05,
  num_leaves       = 31,
  max_depth        = 6,
  feature_fraction = 0.8,
  bagging_fraction = 0.8,
  bagging_freq     = 1
)
lgb_l2_model <- lgb.train(
  params                = params_l2,
  data                  = dtrain_l2,
  nrounds               = 100,
  valids                = list(train = dtrain_l2, val = dval_l2),
  early_stopping_rounds = 10,
  verbose               = 0
)
# Evaluate Layer 2 performance
p_l2_test <- predict(lgb_l2_model, as.matrix(test_l2_df[, feat_names]))
auc_l2_test <- as.numeric(pROC::roc(y_ts_l2, p_l2_test)$auc)
cm_l2 <- confusionMatrix(factor(ifelse(p_l2_test >= 0.5, 1, 0)), factor(y_ts_l2))
rec_l2  <- cm_l2$byClass["Sensitivity"]
spec_l2 <- cm_l2$byClass["Specificity"]
bal_l2  <- cm_l2$byClass["Balanced Accuracy"]
cat("============================================================\n")
cat("   LAYER 2 LIGHTGBM (ESI 2/3 vs ESI 4/5) TEST REPORT\n")
cat("============================================================\n")
cat(sprintf("  ESI 2/3 Sensitivity (Recall): %.4f\n", rec_l2))
cat(sprintf("  ESI 4/5 Specificity         : %.4f\n", spec_l2))
cat(sprintf("  Balanced Accuracy           : %.4f\n", bal_l2))
cat(sprintf("  ROC-AUC (Test)              : %.4f\n", auc_l2_test))
cat("============================================================\n\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Save Model Artifacts to deploy/
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)
# Save Layer 1 LightGBM Model
saveRDS(list(model = lgb_l1_model, preproc = preproc, is_l1_lgb = TRUE), file = file.path(deploy_dir, "lightgbm_layer1_esi1_model.rds"))
cat("Layer 1 LightGBM (ESI 1 Detector) saved to deploy/lightgbm_layer1_esi1_model.rds\n")
# Save Layer 2 LightGBM Model (overwriting rf_esi23_esi45_extreme_model.rds for seamless master pipeline compatibility)
saveRDS(list(model = lgb_l2_model, preproc = preproc, is_lgb_l2_binary = TRUE), file = file.path(deploy_dir, "rf_esi23_esi45_extreme_model.rds"))
cat("Layer 2 LightGBM (ESI 2/3 vs 4/5 Specialist) saved to deploy/rf_esi23_esi45_extreme_model.rds\n")
# Save Layer 1 & 2 Training Summary CSV
reports_dir <- "../reports"
if (!dir.exists(reports_dir)) reports_dir <- "reports"
if (!dir.exists(reports_dir)) dir.create(reports_dir, recursive = TRUE)
l1_l2_report <- data.frame(
  Layer = c("Layer1_ESI1_Detector", "Layer2_ESI23_vs_ESI45_Specialist"),
  Target = c("ESI 1 vs Non-ESI 1", "ESI 2/3 vs ESI 4/5 (No ESI 1)"),
  Recall = c(round(rec_l1, 4), round(rec_l2, 4)),
  Specificity = c(round(spec_l1, 4), round(spec_l2, 4)),
  Balanced_Accuracy = c(round(bal_l1, 4), round(bal_l2, 4)),
  ROC_AUC = c(round(auc_l1_test, 4), round(auc_l2_test, 4))
)
write.csv(l1_l2_report, file = file.path(reports_dir, "hierarchical_l1_l2_training_report.csv"), row.names = FALSE)
cat("Training report written to reports/hierarchical_l1_l2_training_report.csv\n")